# Semana 06 · Restricciones, inferencia y modelos de lenguaje
## Estudio comparativo de tres formas de resolver problemas lógicos
**Equipo: Daniel Salazar, Andretty Ochoa y Jalil Chano · CMP-4004 · 30 de septiembre de 2026**

Estudiamos dónde se encuentra la garantía de una solución: en el modelo formal y el solver, y no automáticamente en la traducción de lenguaje natural.

Conservamos las mediciones originales de Andretty. Ejecutamos nuevamente las pruebas, el horario y el brazo A; para B y C analizamos los CSV guardados. No presentamos ese análisis como una nueva ejecución del LLM.

### Ejecución y evidencia
Se puede leer directamente en GitHub con todas las salidas guardadas. Para ejecutarlo, abrirlo en Jupyter o VS Code y seleccionar **Run All**. No requiere Ollama, GPU ni conexión de red: usa Python, el kernel de Jupyter y los archivos de esta carpeta.

| Evidencia | Procedencia |
|---|---|
| Horario, pruebas y A | Recalculados en este notebook |
| Respuestas de B y C | CSV originales, sin modificaciones |
| Respuestas originales y CSP intermedios de B | No incluidos en la entrega; no podemos auditar sus restricciones ni reproducir sus reintentos históricos |
| Reintentos con respuestas controladas | Pruebas del funcionamiento del código; no son resultados del LLM |

El [reporte](REPORT_WEEK06.md), las [instrucciones del profesor](README.md) y el [registro de IA](../../AI_LOG.md) complementan la presentación.

In [1]:
from pathlib import Path
import sys, csv, json, platform, subprocess, html
from collections import Counter
from IPython.display import display, HTML, Markdown

candidates = [Path.cwd(), Path.cwd() / 'studios/week-06',
              Path.cwd() / 'team-review/studios/week-06']
BASE = next((p.resolve() for p in candidates if (p / 'run_arm_a.py').exists()), None)
assert BASE is not None, 'Abrir el notebook desde el repositorio o desde studios/week-06.'
sys.path.insert(0, str(BASE))
from csp import backtrack_search
from starter import scheduling_csp, run_ablation, ACTIVITIES, SPORTS, LUNCH_AFTER
from logic_lm import load_puzzles, build_csp
from run_arm_a import build_models

def table(headers, rows):
    head = ''.join('<th style="text-align:left;padding:8px">'+html.escape(str(x))+'</th>' for x in headers)
    body = ''.join('<tr>'+''.join('<td style="padding:8px;border-bottom:1px solid #bbb">'+html.escape(str(x))+'</td>' for x in row)+'</tr>' for row in rows)
    display(HTML('<table><thead><tr>'+head+'</tr></thead><tbody>'+body+'</tbody></table>'))

bank = load_puzzles()
puzzles = {p['id']: p for p in bank if not p.get('worked_example')}
print('Python de verificación:', platform.python_version())
print('Banco:', len(bank), '| Ejemplos reservados: 5 | Evaluación:', len(puzzles))

Python de verificación: 3.12.10
Banco: 20 | Ejemplos reservados: 5 | Evaluación: 15


## 1 · Convertimos un horario en un CSP
Un **CSP** tiene variables, dominios y restricciones. Cada actividad es una variable; su valor es el período asignado.

- Variables: Math, English, Science, Sports, Art y Music.
- Dominios: períodos 1–6; para Sports solamente 1–3, antes del almuerzo.
- Restricción global: todas las actividades ocupan períodos distintos.

Representamos `all-different` con un grafo completo de vecinos: cualquier par de actividades debe tener valores distintos. Restringimos Sports antes de buscar, eliminando desde el modelo las opciones inválidas.

In [2]:
schedule = scheduling_csp()
solution, counter = backtrack_search(schedule, use_fc=True, use_mrv=True)
assert set(solution) == set(ACTIVITIES)
assert len(set(solution.values())) == len(ACTIVITIES)
assert all(solution[a] in schedule.domains[a] for a in ACTIVITIES)
assert solution[SPORTS] <= LUNCH_AFTER
table(['Período', 'Actividad', 'Momento'], [(p, a, 'Mañana' if p <= LUNCH_AFTER else 'Tarde')
      for a, p in sorted(solution.items(), key=lambda item: item[1])])
print('Llamadas de backtracking:', counter['calls'])

Período,Actividad,Momento
1,Sports,Mañana
2,Math,Mañana
3,English,Mañana
4,Science,Tarde
5,Art,Tarde
6,Music,Tarde


Llamadas de backtracking: 7


### Ablación: ¿qué aporta cada técnica?
**Backtracking** prueba asignaciones y retrocede al encontrar contradicciones. **Forward checking (FC)** elimina de los vecinos valores incompatibles con la asignación actual. **MRV** elige primero la variable con menos valores disponibles. **LCV** prueba primero el valor que elimina menos opciones de los vecinos. **AC-3** propaga consistencia de arcos antes de iniciar la búsqueda en esta configuración.

LCV y AC-3 se añaden **por separado a FC+MRV**. No son dos pasos consecutivos de una única configuración acumulativa.

In [3]:
run_ablation(scheduling_csp)

  configuration                calls      time   solved
  -----------------------------------------------------
  plain backtracking               8   0.0000s      yes
  + forward checking               7   0.0000s      yes
  + MRV                            7   0.0000s      yes
  + LCV                            7   0.0000s      yes
  + AC-3 preprocessing             7   0.0001s      yes


En este horario pequeño, FC reduce las llamadas de 8 a 7; las demás técnicas no reducen más ese contador. No inferimos ventajas de velocidad a partir de diferencias de fracciones de milisegundo en una única ejecución.

### Verificación con las pruebas originales
El Sudoku HARD permite ver un problema más exigente: backtracking básico agota el presupuesto; FC+MRV+AC-3 lo resuelve. LCV aumenta las llamadas en este caso. Que una heurística sea razonable no significa que siempre ayude.

In [4]:
for script in ['test_csp.py', '_verify_puzzles.py', 'test_refinement.py']:
    result = subprocess.run([sys.executable, str(BASE / script)], cwd=BASE,
                            capture_output=True, text=True, encoding='utf-8')
    print(f'--- {script} ---')
    print(result.stdout + result.stderr)
    assert result.returncode == 0, f'Falló {script}'

--- test_csp.py ---
  ok  schedule is a complete, all-different timetable (6 activities)
  ok  sports lands in period 1 (<= 3, before lunch)
  ok  AC-3 shrinks HARD domains 593 -> 320 (273 values eliminated before any guess)
  ok  plain backtracking TIMES OUT on HARD; fc+mrv+ac3 solves in 34,478 calls (the guarantee fails on request)
  ok  LCV HURTS on HARD: 34,478 -> 56,813 backtrack calls (a heuristic gamble that did not pay back)

all 5 tests pass

--- _verify_puzzles.py ---
  ok  p01  unique + matches gold [worked]
  ok  p02  unique + matches gold [worked]
  ok  p03  unique + matches gold [worked]
  ok  p04  unique + matches gold [worked]
  ok  p05  unique + matches gold [worked]
  ok  p06  unique + matches gold
  ok  p07  unique + matches gold
  ok  p08  unique + matches gold
  ok  p09  unique + matches gold
  ok  p10  unique + matches gold
  ok  p11  unique + matches gold
  ok  p12  unique + matches gold
  ok  p13  unique + matches gold
  ok  p14  unique + matches gold
  ok  p15 

## 2 · Mismo banco, tres pipelines
| Brazo | Construcción de la respuesta | Qué evaluamos |
|---|---|---|
| A | Modelos explícitos → solver CSP | Control formal asistido |
| B | LLM → modelo CSP → solver | Traducción y refinamiento |
| C | LLM → respuesta directa | Resolución sin solver |

Usamos p06–p20; p01–p05 son ejemplos. La ejecución original utilizó **qwen2.5:3b, Ollama 0.34.4, CPU, Python 3.13.2**, con `temperature=0`, `seed=42`, `num_ctx=4096` y `num_predict=2048`.

**Alcance del control A:** construimos los modelos con asistencia de IA y acceso al banco de referencia. El programa no utiliza `spec` ni `gold` para construirlos, pero no se trata de modelado humano independiente a ciegas.

**Comparación B/C:** B recibe un ejemplo de traducción y C un ejemplo de formato. Por tanto, comparamos estos pipelines concretos; no aislamos causalmente el efecto del solver.

In [5]:
models = build_models()
records_a = []
for pid, puzzle in puzzles.items():
    answer, count = backtrack_search(build_csp(models[pid]), use_fc=True, use_mrv=True, limit=200_000)
    assert answer == puzzle['gold'], pid
    records_a.append([pid, 'correct', count['calls'], json.dumps(answer, ensure_ascii=False)])
table(['Acertijo', 'Resultado reproducido A', 'Llamadas', 'Asignación'], records_a)

Acertijo,Resultado reproducido A,Llamadas,Asignación
p06,correct,3,"{""black"": 1, ""red"": 2, ""silver"": 3}"
p07,correct,3,"{""Ben"": 1, ""Dee"": 4, ""Ada"": 3, ""Cy"": 2}"
p08,correct,3,"{""A"": ""white"", ""B"": ""gray"", ""C"": ""white""}"
p09,correct,3,"{""Novel"": 1, ""Essays"": 3, ""Comic"": 5, ""Poems"": 4, ""Manual"": 2}"
p10,correct,3,"{""Dishes"": ""Mon"", ""Vacuum"": ""Thu"", ""Trash"": ""Wed"", ""Laundry"": ""Tue""}"
p11,correct,2,"{""X"": ""plum"", ""Y"": ""teal"", ""Z"": ""olive""}"
p12,correct,4,"{""R"": 2, ""S"": 4, ""Q"": 3, ""P"": 1}"
p13,correct,3,"{""Al"": 1, ""Cy"": 2, ""Ed"": 5, ""Bo"": 4, ""Di"": 3}"
p14,correct,3,"{""Lamp1"": ""on"", ""Lamp2"": ""off"", ""Lamp3"": ""on""}"
p15,correct,3,"{""N"": ""red"", ""S"": ""green"", ""E"": ""yellow"", ""W"": ""blue""}"


### B · Refinamiento dirigido por errores
1. Solicitamos una traducción a JSON con variables, dominios y restricciones `alldiff`, `neq` y `eq`.
2. El clasificador intenta construir y resolver el CSP.
3. Si el JSON o su estructura son inválidos, devolvemos información del error. Si no existe solución, solicitamos revisar contradicciones.
4. Permitimos hasta **tres intentos totales**, incluido el inicial.

No reintentamos porque la respuesta difiera de `gold`: esa respuesta de referencia sirve para evaluar después, no para ayudar al modelo a acertar. Tampoco reintentamos un timeout del solver. Sin/con refinamiento comparten la primera respuesta mediante la caché en la ejecución original.

| Categoría | Interpretación |
|---|---|
| `malformed_json` | JSON inválido o estructura CSP inutilizable |
| `valid_json_wrong_model` | El solver encuentra una asignación que no coincide con gold |
| `valid_no_solution` | El CSP traducido es insatisfacible |
| `timeout` | Se supera el límite de llamadas del solver; no es un timeout de red |

Una discrepancia con gold puede ser un error de contenido o de representación. Una traducción incorrecta también puede ser satisfacible: el solver no detecta automáticamente todos los errores del LLM.

## 3 · Resultados originales y consistencia de los archivos
Calculamos las cifras directamente desde los CSV, conservando la evaluación estricta original. Verificamos que cada archivo contenga exactamente los 15 identificadores esperados y que las etiquetas coincidan con las respuestas guardadas cuando estas permiten comprobarlas.

In [6]:
data = {}
for arm in 'abc':
    with (BASE / f'results_arm_{arm}.csv').open(encoding='utf-8', newline='') as f:
        data[arm] = list(csv.DictReader(f))
    assert len(data[arm]) == 15
    assert {r['id'] for r in data[arm]} == set(puzzles)
for arm, answer_field in [('a', 'solution'), ('b', 'solution'), ('c', 'answer')]:
    for row in data[arm]:
        answer = json.loads(row[answer_field])
        matches = isinstance(answer, dict) and all(answer.get(k) == v for k, v in puzzles[row['id']]['gold'].items())
        label = row['with_refinement'] if arm == 'b' else row['category']
        assert matches == (label == 'correct'), (arm, row['id'])

labels = ['A', 'B sin refinamiento', 'B con refinamiento', 'C']
counts = [sum(r['category']=='correct' for r in data['a']),
          sum(r['without_refinement']=='correct' for r in data['b']),
          sum(r['with_refinement']=='correct' for r in data['b']),
          sum(r['category']=='correct' for r in data['c'])]
table(['Pipeline', 'Aciertos', 'Tasa estricta'], [(label, f'{n}/15', f'{n/15:.1%}') for label,n in zip(labels,counts)])
bars = []
for label,n in zip(labels,counts):
    bars.append(f'<div style="margin:12px 0"><b>{label}: {n}/15</b><div style="background:#e2e8f0;height:22px;width:100%"><div style="background:#2563eb;height:22px;width:{100*n/15}%"></div></div></div>')
display(HTML('<div style="max-width:650px">'+''.join(bars)+'</div>'))
print('Cambio por refinamiento:', f'{100*(counts[2]-counts[1])/15:.1f}', 'puntos porcentuales')

Pipeline,Aciertos,Tasa estricta
A,15/15,100.0%
B sin refinamiento,7/15,46.7%
B con refinamiento,7/15,46.7%
C,7/15,46.7%


Cambio por refinamiento: 0.0 puntos porcentuales


In [7]:
before = Counter(r['without_refinement'] for r in data['b'])
after = Counter(r['with_refinement'] for r in data['b'])
categories = ['correct', 'malformed_json', 'valid_json_wrong_model', 'valid_no_solution', 'timeout']
table(['Categoría B', 'Sin refinamiento', 'Con refinamiento'], [(c,before[c],after[c]) for c in categories])
table(['Acertijo', 'B sin', 'B con', 'Intentos'], [(r['id'],r['without_refinement'],r['with_refinement'],r['attempts']) for r in data['b']])

Categoría B,Sin refinamiento,Con refinamiento
correct,7,7
malformed_json,0,0
valid_json_wrong_model,7,7
valid_no_solution,1,1
timeout,0,0


Acertijo,B sin,B con,Intentos
p06,valid_json_wrong_model,valid_json_wrong_model,1
p07,valid_json_wrong_model,valid_json_wrong_model,1
p08,valid_json_wrong_model,valid_json_wrong_model,1
p09,correct,correct,1
p10,correct,correct,1
p11,correct,correct,1
p12,correct,correct,1
p13,correct,correct,1
p14,correct,correct,1
p15,valid_json_wrong_model,valid_json_wrong_model,1


**Lectura:** A alcanza 15/15. B y C empatan en 7/15 bajo el criterio estricto. En B, p17 es el único caso con tres intentos; permanece sin solución. Los otros desacuerdos no activan refinamiento, por lo que no observamos mejora.

Estos CSV permiten verificar conteos y asignaciones finales, pero no reconstruir los CSP intermedios de B ni certificar su fidelidad al enunciado. Para ello necesitamos las respuestas originales de `.llm_cache`, ausentes en el repositorio.

## 4 · ¿Error lógico o diferencia de representación?
La evaluación estricta compara nombres y valores con gold; distingue, por ejemplo, `2` de `"2"`. En B y C no rechaza claves adicionales si todas las esperadas coinciden. Mantenemos esa métrica y añadimos una **auditoría posterior explícita**, sin cambiar los CSV.

Aplicamos únicamente las equivalencias identificadas en el reporte: en B, p06 (`black_car`/`spot1`), p08 (`HouseA`) y p16 (escritorio→persona); en C, cadenas numéricas en p06/p07/p09/p12 y días completos en p10. Estas transformaciones no consultan gold para elegir una respuesta; gold se usa después para comparar.

In [8]:
def normalize(arm, pid, answer):
    answer = dict(answer) if isinstance(answer, dict) else None
    if answer is None:
        return None
    if arm == 'b' and pid == 'p06':
        return {k.removesuffix('_car'): int(v.removeprefix('spot')) for k,v in answer.items()}
    if arm == 'b' and pid == 'p08':
        return {k.removeprefix('House'): v for k,v in answer.items()}
    if arm == 'b' and pid == 'p16':
        assert len(set(answer.values())) == len(answer)
        return {v:k for k,v in answer.items()}
    if arm == 'c' and pid in {'p06','p07','p09','p12'}:
        return {k:int(v) for k,v in answer.items()}
    if arm == 'c' and pid == 'p10':
        days = {'Monday':'Mon', 'Tuesday':'Tue', 'Wednesday':'Wed', 'Thursday':'Thu'}
        return {k:days.get(v,v) for k,v in answer.items()}
    return answer

audit_rows = []
for arm, field in [('b','solution'),('c','answer')]:
    passed, extra = [], []
    for row in data[arm]:
        original = json.loads(row[field])
        revised = normalize(arm, row['id'], original)
        if revised == puzzles[row['id']]['gold']:
            passed.append(row['id'])
            if original != revised:
                extra.append(row['id'])
    audit_rows.append([arm.upper(), ', '.join(extra), f'{len(passed)}/15', f'{len(passed)/15:.1%}'])
table(['Brazo', 'Equivalencias adicionales', 'Asignaciones correctas tras revisión', 'Tasa'], audit_rows)

Brazo,Equivalencias adicionales,Asignaciones correctas tras revisión,Tasa
B,"p06, p08, p16",10/15,66.7%
C,"p06, p07, p09, p10, p12",12/15,80.0%


Tras esa revisión obtenemos **B: 10/15 (66.7 %) y C: 12/15 (80 %)**. Son asignaciones equivalentes en significado; no demostramos que los modelos formales generados por B contengan todas las restricciones correctas.

### Análisis de dos casos
- **B, p07:** Ben y Cy aparecen ambos en el período 1. La asignación viola la unicidad del problema original, aunque el solver haya podido resolver el modelo recibido. Sin el CSP generado no atribuimos el fallo a una restricción específica.
- **C, p06:** devuelve las posiciones correctas como cadenas (`"2"`, `"3"`, `"1"`). El criterio estricto lo marca incorrecto, pero el contenido de la asignación es correcto.

## 5 · Limitaciones y conclusión
1. El banco es pequeño: 15 acertijos de evaluación y un único modelo/configuración. No generalizamos a todos los LLM.
2. Ajustamos el prompt de B después de observar errores en una ejecución parcial de evaluación. No es un conjunto de prueba completamente intacto.
3. A tuvo asistencia y acceso a las referencias; los prompts de B y C tampoco están controlados para aislar una sola variable.
4. La auditoría de equivalencias es posterior. No sustituye una regla de normalización definida antes del experimento.
5. No medimos tiempos globales ni esfuerzo humano de A/B/C. La caché original de B no está adjunta.
6. La corrección posterior del manejo de estructuras inválidas se comprueba con respuestas controladas; no atribuimos nuevos resultados experimentales a esa corrección.

**Conclusión:** los modelos explícitos de A resolvieron los 15 casos. El refinamiento de B no mejoró la tasa final en esta ejecución. C obtuvo más asignaciones semánticamente correctas en la auditoría posterior. El solver garantiza el cumplimiento del modelo formal que recibe bajo sus condiciones; la fidelidad de la traducción sigue siendo un problema de verificación.

**Siguiente experimento:** conservar todas las traducciones y reintentos, definir un esquema de salida común, congelar los prompts antes de evaluar y usar un conjunto nuevo para comparar con y sin refinamiento.

## Apéndice · Fundamentos técnicos

**¿Por qué usar un grafo completo para el horario?** Porque cada par de actividades debe diferir. Con seis variables hay 15 pares distintos; los conjuntos de vecinos representan cada relación en ambos sentidos.

**¿Por qué el horario necesita siete llamadas con FC?** Hay seis asignaciones y la llamada final que reconoce una asignación completa. El contador mide entradas a la función recursiva, no solamente retrocesos fallidos.

**¿Qué significa que backtracking sea completo?** Con dominios finitos y sin un corte que interrumpa la exploración, encuentra una solución si existe o demuestra que no existe. Nuestro límite de llamadas puede detenerlo antes: `TIMEOUT` no demuestra insatisfacibilidad.

**¿FC y AC-3 son lo mismo?** No. FC elimina valores incompatibles en los vecinos de la variable recién asignada. AC-3 revisa arcos repetidamente y propaga eliminaciones. En este solver, `use_ac3=True` realiza preprocesamiento, no mantenimiento de consistencia de arcos en cada nodo.

**¿Qué garantiza AC-3?** Consistencia de arcos para las restricciones binarias compatibles con su implementación. No garantiza por sí sola resolver cualquier CSP ni demostrar toda insatisfacibilidad. El motor proporcionado usa desigualdad entre vecinos; no debe generalizarse sin cambios a restricciones binarias arbitrarias.

**¿Por qué MRV suele ayudar?** Ataca primero las variables con menos opciones para descubrir contradicciones pronto. En empates se usa el grado en este motor. No garantiza ser mejor en toda instancia.

**¿Por qué LCV empeora HARD?** Ordenar valores para restringir menos a los vecinos puede explorar primero ramas poco útiles. En la prueba las llamadas pasan de 34 478 a 56 813; además calcular el orden tiene un costo. Es una observación de esa instancia, no una regla universal.

**¿B puede devolver algo incorrecto aunque el solver funcione?** Sí: el solver resuelve el CSP traducido, que puede omitir o interpretar mal restricciones. También puede haber diferencias de representación respecto a gold.

**¿Por qué no corregimos B usando gold?** Eso filtraría la respuesta de referencia al proceso de resolución. Gold sirve para evaluación externa. Los reintentos reciben errores de estructura o ausencia de solución.

**¿Por qué el refinamiento dio cero mejora?** Siete discrepancias se clasificaron como modelos con solución diferente, categoría que no activa un reintento. p17 sí reintentó, pero siguió sin solución. No concluimos que el refinamiento nunca funcione.

**¿Tres reintentos o tres intentos?** La implementación usa `range(1, max_retries + 1)`: tres intentos totales, como máximo dos correcciones posteriores al inicial.

**¿El 100 % de A demuestra superioridad humana?** No. Es un control formal asistido, desarrollado con acceso al banco. Confirma que esos modelos resuelven estos casos; no mide capacidad humana independiente ni tiempo de modelado.

**¿C ganó?** En la métrica estricta empata con B. En la auditoría posterior de asignaciones alcanza 12/15 frente a 10/15. Solo podemos afirmar eso sobre esta configuración y estos datos, con las limitaciones del diseño.

**¿La auditoría de representación está cambiando resultados para favorecer un brazo?** Conservamos el resultado original y declaramos cada transformación. Es un análisis posterior que explica discrepancias, no una métrica predefinida ni un reemplazo de la evaluación principal.

**¿Por qué cinco ejemplos y quince casos?** El banco tiene veinte y el profesor marca cinco como ejemplos trabajados. Evaluamos los otros quince. En el prompt final de B utilizamos p01 como ejemplo completo.

**¿Podemos repetir exactamente B y C?** Los CSV permiten repetir el análisis. Para reproducir las respuestas históricas de B faltan los archivos de caché. Una nueva llamada con el mismo nombre de modelo y semilla no garantiza identidad entre versiones o entornos.

**¿Qué comprobamos sin ejecutar Ollama?** Las cinco pruebas originales, los veinte acertijos, el horario, el brazo A, la coherencia de los CSV, las equivalencias declaradas y cuatro pruebas de refinamiento con respuestas controladas.

**¿Qué asistencia utilizamos?** Está declarada en [AI_LOG.md](../../AI_LOG.md): implementación, diagnóstico, análisis, redacción y preparación del notebook. La asistencia está permitida por el curso; debemos comprender y poder defender las decisiones.

### Archivos y fuentes
- [Instrucciones del estudio](README.md) · [Reporte original con actualización de verificación](REPORT_WEEK06.md)
- [Horario y refinamiento](starter.py) · [Modelos de A](run_arm_a.py) · [Cliente y ejecución B](run_experiment.py) · [Ejecución C](run_arm_c.py)
- [Resultados A](results_arm_a.csv) · [Resultados B](results_arm_b.csv) · [Resultados C](results_arm_c.csv)
- [Material del profesor](https://github.com/aproano2/cmp-4004-fall26/tree/main/studios/week-06) · [Política de IA](https://github.com/aproano2/cmp-4004-fall26/blob/main/resources/ai-policy.md)